## Eager-Only Solution

> 📖 Read the full article: [Writing Portable DataFrame Code for Eager and Lazy Execution](https://codecut.ai/eager-to-lazy-dataframes-with-narwhals/)


First, we’ll build a fill-forward solution compatible with eager pandas and Polars dataframes.

Suppose we have a dataset of store sales over time:

In [ ]:
from datetime import datetime

data = {
    'sale_date': [
        datetime(2025, 5, 22),
        datetime(2025, 5, 23),
        datetime(2025, 5, 24),
        datetime(2025, 5, 22),
        datetime(2025, 5, 23),
        datetime(2025, 5, 24),
    ],
    'store': [
        'Thimphu',
        'Thimphu',
        'Thimphu',
        'Paro',
        'Paro',
        'Paro',
    ],
    'sales': [1100, None, 1450, 501, 500, None],
}

Note that for the ‘sales’ column, we have some missing values. Let’s write a dataframe-agnostic function to forward-fill missing prices using the previous valid value from the same store.

To avoid locking ourselves into either pandas or Polars, we’ll use [Narwhals](https://github.com/narwhals-dev/narwhals) (see [our previous post on Narwhals](https://codecut.ai/unified-dataframe-functions-pandas-polars-pyspark/) for how to get started with it):

In [ ]:
import narwhals as nw
from narwhals.typing import IntoFrameT


def agnostic_ffill_by_store(df_native: IntoFrameT) -> IntoFrameT:
    return (
        nw.from_native(df_native)
        .with_columns(nw.col("sales").fill_null(strategy="forward").over("store"))
        .to_native()
    )

You can verify that this works for both `pandas.DataFrame` and `polars.DataFrame`:

In [ ]:
import polars as pl
import pandas as pd
import pyarrow as pa

# pandas.DataFrame
df_pandas = pd.DataFrame(data)
agnostic_ffill_by_store(df_pandas)

Output:


In [ ]:
# polars.DataFrame
df_polars = pl.DataFrame(data)
print(agnostic_ffill_by_store(df_polars))

## Eager and Lazy Solution

But what if we also want it to support `polars.LazyFrame`, `duckdb.DuckDBPyRelation`, and `pyspark.sql.DataFrame`? If we try passing any of those to `agnostic_ffill_by_store`, then Narwhals will raise an error:

In [ ]:
import duckdb

duckdb_rel = duckdb.table('df_polars')
agnostic_ffill_by_store(duckdb_rel)

This error happens because lazy DataFrames don’t preserve the order of rows unless explicitly told to do so. Thus, operations like `fill_null(strategy='forward')` may yield unpredictable results unless you specify an ordering column explicitly.

To fix this, we must explicitly define how the rows should be ordered. Since our dataset has a `sale_date` column that represents time, it’s a natural choice for ordering forward-fill operations.

In [ ]:
def agnostic_ffill_by_store_improved(df_native: IntoFrameT) -> IntoFrameT:
    return (
        nw.from_native(df_native)
        .with_columns(
            nw.col("sales")
            .fill_null(strategy="forward")
            # Note the `order_by` statement
            .over("store", order_by="sale_date")
        )
        .to_native()
    )

And now – voilà – it also supports DuckDB and Polars LazyFrame!

In [ ]:
agnostic_ffill_by_store_improved(duckdb_rel)

In [ ]:
agnostic_ffill_by_store_improved(df_polars.lazy()).collect()

Being forced to specify an ordering column for `fill_null(strategy='forward')` may feel like a restriction on the Polars API. However, if we want to write portable and maintainable code, we cannot make assumptions about whether our backend preserves row order, and so this is a restriction that enables portability!